In [ ]:
# =============================================================================
# CELL 1: Setup & Basic Prompt Templates with Variables
# =============================================================================
# This notebook examines different approaches to reusable prompts - similar to 
# parameterized SQL queries but for prompts. Define the structure once,
# swap in different data each time.
# =============================================================================

import boto3
import json
import re
from datetime import datetime
from string import Template

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

def ask(prompt, system=None, max_tokens=2048, temperature=0.0):
    kwargs = {
        'modelId': model_id,
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system:
        kwargs['system'] = [{'text': system}]
    response = bedrock.converse(**kwargs)
    return {
        'text': response['output']['message']['content'][0]['text'],
        'input_tokens': response['usage']['inputTokens'],
        'output_tokens': response['usage']['outputTokens'],
        'stop_reason': response['stopReason']
    }

def parse_llm_json(text):
    """Strip markdown fencing and parse JSON from LLM output."""
    raw = text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

# --- Approach 1: Python f-strings (what we've been doing) ---
print("=" * 70)
print("APPROACH 1: F-String Templates (Simple but Limited)")
print("=" * 70)

claim_type = "auto"
claim_text = "Rear-ended at a red light. Bumper damage, $3,200 estimate."
output_format = "two sentences"

fstring_prompt = f"Classify this {claim_type} insurance claim and summarize in {output_format}.\n\nCLAIM: {claim_text}"

result = ask(fstring_prompt)
print(f"Prompt: {fstring_prompt}")
print(f"Answer: {result['text']}")
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Approach 2: Python string.Template (safer, reusable) ---
print("\n" + "=" * 70)
print("APPROACH 2: string.Template (Safer Variable Substitution)")
print("=" * 70)

classification_template = Template("""Classify this $claim_type insurance claim.

CLAIM: $claim_text

Respond in EXACTLY this format:
Classification: [type]
Confidence: $confidence_options
Reason: [one sentence]""")

# Use it with different data
claims = [
    {
        'claim_type': 'auto',
        'claim_text': 'Deer ran into the road, driver swerved and hit a guardrail. Front end damage.',
        'confidence_options': 'High/Medium/Low'
    },
    {
        'claim_type': 'homeowners',
        'claim_text': 'Lightning struck a tree in the backyard which fell onto the garage roof.',
        'confidence_options': 'High/Medium/Low'
    },
    {
        'claim_type': 'auto',
        'claim_text': 'Car was parked overnight and owner found a cracked windshield in the morning. No witnesses.',
        'confidence_options': 'High/Medium/Low'
    }
]

for i, claim in enumerate(claims, 1):
    prompt = classification_template.substitute(claim)
    result = ask(prompt)
    print(f"\nClaim {i}: {claim['claim_text'][:60]}...")
    print(f"Result: {result['text']}")

# --- Approach 3: Dictionary-based template engine ---
print("\n" + "=" * 70)
print("APPROACH 3: Template Engine Class (Production Pattern)")
print("=" * 70)

class PromptTemplate:
    """
    A reusable prompt template with named variables and validation.
    Think of it like a parameterized SQL query — the structure is fixed,
    the data changes each time.
    """
    
    def __init__(self, name, template_str, required_vars, 
                 system_template=None, defaults=None):
        self.name = name
        self.template = Template(template_str)
        self.required_vars = required_vars
        self.system_template = Template(system_template) if system_template else None
        self.defaults = defaults or {}
    
    def render(self, **kwargs):
        """Fill in the template with provided variables."""
        # Apply defaults for missing optional vars
        merged = {**self.defaults, **kwargs}
        
        # Validate required vars
        missing = [v for v in self.required_vars if v not in merged]
        if missing:
            raise ValueError(f"Template '{self.name}' missing required vars: {missing}")
        
        prompt = self.template.substitute(merged)
        system = self.system_template.substitute(merged) if self.system_template else None
        
        return prompt, system
    
    def run(self, max_tokens=2048, temperature=0.0, **kwargs):
        """Render the template and call the LLM."""
        prompt, system = self.render(**kwargs)
        result = ask(prompt, system=system, max_tokens=max_tokens, 
                     temperature=temperature)
        result['template_name'] = self.name
        result['variables'] = kwargs
        return result

# --- Build a template ---
claim_summary_template = PromptTemplate(
    name="claim_summary",
    template_str="""Summarize this $claim_type insurance claim for a $audience.

CLAIM:
$claim_text

Provide your summary in $output_format.""",
    required_vars=['claim_type', 'claim_text'],
    system_template="""You are a $role at $company. 
Keep responses concise and appropriate for the intended audience.""",
    defaults={
        'audience': 'claims supervisor',
        'output_format': 'two sentences',
        'role': 'senior claims adjuster',
        'company': 'Evergreen Mutual Insurance'
    }
)

# Run with minimal variables (defaults fill the rest)
print("\nRun 1 — Minimal variables (defaults handle the rest):")
result = claim_summary_template.run(
    claim_type='auto',
    claim_text='Insured rear-ended at highway speed. Significant trunk damage, airbags deployed. Driver transported to ER with back pain.'
)
print(f"Template: {result['template_name']}")
print(f"Answer: {result['text']}")
print(f"[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# Run with overridden defaults
print("\nRun 2 — Override defaults for a different audience:")
result = claim_summary_template.run(
    claim_type='auto',
    claim_text='Insured rear-ended at highway speed. Significant trunk damage, airbags deployed. Driver transported to ER with back pain.',
    audience='the policyholder',
    output_format='a brief, reassuring paragraph',
    role='customer service representative'
)
print(f"Template: {result['template_name']}")
print(f"Answer: {result['text']}")
print(f"[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# Run for a completely different claim
print("\nRun 3 — Different claim, same template:")
result = claim_summary_template.run(
    claim_type='homeowners',
    claim_text='Kitchen fire started from unattended stove. Fire department responded. Significant smoke and water damage throughout first floor. Family displaced.',
    audience='underwriting review committee',
    role='senior claims adjuster'
)
print(f"Template: {result['template_name']}")
print(f"Answer: {result['text']}")
print(f"[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Show the progression ---
print("\n" + "=" * 70)
print("TEMPLATE APPROACHES — PROGRESSION")
print("=" * 70)
print("""
  F-STRINGS:
    ✓ Simple, everyone knows them
    ✗ No validation, no defaults, no reusability
    → Good for: prototyping, one-off prompts

  string.Template:
    ✓ Safer substitution ($var not evaluated as code)
    ✓ Reusable across different data
    ✗ No validation, no defaults
    → Good for: simple repeatable tasks

  PromptTemplate class:
    ✓ Named templates with validation
    ✓ Required vars checked before LLM call
    ✓ Defaults for optional vars
    ✓ System prompt templated too
    ✓ Metadata tracking (which template, which vars)
    → Good for: production systems
""")

In [ ]:
# =============================================================================
# Template Library — Reusable Insurance Task Templates
# =============================================================================
# A library of production-ready templates for common insurance tasks.
# Each template is self-contained with its own system prompt, user prompt,
# required variables, and defaults. Pick a template, pass your data, go.
# =============================================================================

# --- Template Library ---
TEMPLATE_LIBRARY = {}

def register_template(template):
    """Register a template in the library."""
    TEMPLATE_LIBRARY[template.name] = template
    return template

# --- Template 1: Claim Classification ---
register_template(PromptTemplate(
    name="classify_claim",
    system_template="""You are a claims intake specialist at $company. 
You classify incoming claims accurately and consistently.""",
    template_str="""Classify this $line_of_business insurance claim.

CLAIM:
$claim_text

Return ONLY valid JSON, no markdown fencing:
{
  "classification": "string",
  "sub_type": "string",
  "confidence": "HIGH | MEDIUM | LOW",
  "reason": "string (one sentence)",
  "priority": "URGENT | HIGH | MEDIUM | LOW",
  "routing": "string (which department or team)"
}""",
    required_vars=['claim_text'],
    defaults={
        'line_of_business': 'auto',
        'company': 'Evergreen Mutual Insurance'
    }
))

# --- Template 2: Coverage Determination ---
register_template(PromptTemplate(
    name="coverage_determination",
    system_template="""You are a senior $line_of_business claims adjuster at $company 
with deep expertise in policy interpretation.

GUARDRAILS:
- Ground analysis in the policy details provided
- Flag missing information rather than assuming
- Include disclaimer: "Preliminary analysis — subject to formal review"
""",
    template_str="""Analyze coverage for this claim using step-by-step reasoning.

POLICY DETAILS:
$policy_details

CLAIM:
$claim_text

<reasoning>
1. Identify the cause of loss and classify the peril
2. Verify applicable coverage section
3. Review exclusions that might apply
4. Assess policyholder compliance with conditions
5. Calculate covered amount against limits and deductible
6. Flag items needing investigation
</reasoning>

<determination>
Provide your preliminary coverage determination.
</determination>

Return a JSON summary after your analysis (no markdown fencing):
<claim_json>
{
  "peril": "string",
  "coverage_applies": true/false,
  "exclusions_triggered": ["string"],
  "covered_amount": number,
  "deductible": number,
  "payable": number,
  "investigation_items": ["string"],
  "confidence": "HIGH | MEDIUM | LOW"
}
</claim_json>""",
    required_vars=['claim_text', 'policy_details'],
    defaults={
        'line_of_business': 'homeowners',
        'company': 'Evergreen Mutual Insurance'
    }
))

# --- Template 3: Customer Communication ---
register_template(PromptTemplate(
    name="customer_email",
    system_template="""You are $agent_name, a $role at $company. 
You write clear, empathetic, professional emails to policyholders.

GUARDRAILS:
- Never guarantee coverage outcomes
- Never provide legal advice
- Never discuss internal reserves or processes
- Always include next steps for the policyholder
""",
    template_str="""Write a $email_type email to the policyholder.

POLICYHOLDER: $policyholder_name
CLAIM NUMBER: $claim_number
CLAIM STATUS: $claim_status

CONTEXT:
$context

TONE: $tone
LENGTH: $length""",
    required_vars=['policyholder_name', 'claim_number', 'claim_status', 'context'],
    defaults={
        'email_type': 'status update',
        'tone': 'professional but warm',
        'length': '150-200 words',
        'agent_name': 'Alex Rivera',
        'role': 'claims specialist',
        'company': 'Evergreen Mutual Insurance'
    }
))

# --- Template 4: Fraud Screening ---
register_template(PromptTemplate(
    name="fraud_screening",
    system_template="""You are a Special Investigations Unit analyst at $company.
You screen claims for fraud indicators using evidence-based analysis.

METHODOLOGY:
- Evaluate each red flag independently
- Assign a risk score based on cumulative indicators
- Never accuse — only flag for investigation
- Document your reasoning for each indicator""",
    template_str="""Screen this $line_of_business claim for fraud indicators.

CLAIM:
$claim_text

CLAIMANT HISTORY:
$claimant_history

Check for these common indicators:
- Timing patterns (recent policy changes, reported delays)
- Inconsistencies in the narrative
- Financial stress indicators
- Pattern matching with known schemes
- Documentation gaps

Return ONLY valid JSON, no markdown fencing:
{
  "risk_score": number (1-10),
  "risk_level": "LOW | MODERATE | HIGH | CRITICAL",
  "indicators_found": [
    {"indicator": "string", "severity": "LOW|MEDIUM|HIGH", "evidence": "string"}
  ],
  "recommended_action": "CLEAR | MONITOR | SIU_REVIEW | SIU_PRIORITY",
  "reasoning": "string (2-3 sentences)"
}""",
    required_vars=['claim_text'],
    defaults={
        'line_of_business': 'auto',
        'claimant_history': 'No prior claims on file.',
        'company': 'Evergreen Mutual Insurance'
    }
))

# --- Show the library ---
print("=" * 70)
print("TEMPLATE LIBRARY")
print("=" * 70)
for name, tmpl in TEMPLATE_LIBRARY.items():
    print(f"\n  {name}")
    print(f"    Required: {tmpl.required_vars}")
    print(f"    Defaults: {list(tmpl.defaults.keys())}")

# --- Demo: Run each template ---
sample_claim = """On February 28, 2026, the insured reports that a pipe burst in 
their upstairs bathroom while they were at work. Water damaged the bathroom floor, 
hallway ceiling below, and a living room area rug valued at $4,500. Emergency 
plumber responded same day. The pipe was copper, original to the 1998 construction. 
Estimated damage: $18,000 structural, $4,500 personal property. Policy: HO-3, 
$400K dwelling, $200K personal property, $1,500 deductible."""

# Template 1: Classification
print("\n" + "=" * 70)
print("DEMO 1: classify_claim")
print("=" * 70)

result = TEMPLATE_LIBRARY['classify_claim'].run(
    claim_text=sample_claim,
    line_of_business='homeowners'
)
try:
    parsed = parse_llm_json(result['text'])
    print(f"  Classification: {parsed['classification']}")
    print(f"  Priority: {parsed['priority']}")
    print(f"  Routing: {parsed['routing']}")
except:
    print(result['text'])
print(f"  [Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# Template 3: Customer Email
print("\n" + "=" * 70)
print("DEMO 2: customer_email")
print("=" * 70)

result = TEMPLATE_LIBRARY['customer_email'].run(
    policyholder_name='Margaret Holloway',
    claim_number='EM-CLM-2026-03312',
    claim_status='Under Investigation — Adjuster Assigned',
    context='Pipe burst claim filed yesterday. Adjuster visiting property tomorrow at 10 AM. Emergency plumber has completed temporary repairs.',
    email_type='initial acknowledgment'
)
print(result['text'])
print(f"\n  [Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# Template 4: Fraud Screening
print("\n" + "=" * 70)
print("DEMO 3: fraud_screening")
print("=" * 70)

result = TEMPLATE_LIBRARY['fraud_screening'].run(
    claim_text=sample_claim,
    line_of_business='homeowners',
    claimant_history="""Prior claims:
- 2022: Water damage claim, $8,200 paid
- 2024: Theft claim, $3,100 paid
Policy increased from $300K to $400K dwelling coverage 3 months ago."""
)
try:
    parsed = parse_llm_json(result['text'])
    print(f"  Risk Score: {parsed['risk_score']}/10")
    print(f"  Risk Level: {parsed['risk_level']}")
    print(f"  Action: {parsed['recommended_action']}")
    print(f"  Indicators: {len(parsed['indicators_found'])}")
    for ind in parsed['indicators_found']:
        print(f"    • [{ind['severity']}] {ind['indicator']}: {ind['evidence']}")
except:
    print(result['text'])
print(f"\n  [Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

print("\n" + "=" * 70)
print("TEMPLATE LIBRARY BENEFITS")
print("=" * 70)
print(f"""
  Templates registered: {len(TEMPLATE_LIBRARY)}
  
  Any developer on the team can now:
    result = TEMPLATE_LIBRARY['classify_claim'].run(claim_text=claim)
    result = TEMPLATE_LIBRARY['fraud_screening'].run(claim_text=claim)
  
  No prompt engineering knowledge required — just pass the data.
  The template handles persona, scope, guardrails, CoT, and output format.
  
  Adding a new task = registering a new template. That's it.
""")

In [ ]:
# =============================================================================
# Conditional Templates — Dynamic Sections Based on Claim Type
# =============================================================================
# Real claims aren't one-size-fits-all. An auto claim needs vehicle info
# and liability analysis. A homeowners claim needs property details and
# coverage form analysis. The template should adapt its sections based
# on what kind of claim it's processing — like a smart form that shows
# different fields depending on what you select in the first dropdown.
# =============================================================================

class ConditionalTemplate:
    """
    A template that assembles different prompt sections based on conditions.
    Think of it like a dynamic form — the fields change based on the claim type.
    """
    
    def __init__(self, name, base_system, base_prompt, 
                 conditional_sections, output_schema,
                 required_vars, defaults=None):
        self.name = name
        self.base_system = base_system
        self.base_prompt = base_prompt
        self.conditional_sections = conditional_sections
        self.output_schema = output_schema
        self.required_vars = required_vars
        self.defaults = defaults or {}
    
    def build(self, **kwargs):
        """Assemble the prompt with applicable conditional sections."""
        merged = {**self.defaults, **kwargs}
        
        # Validate required vars
        missing = [v for v in self.required_vars if v not in merged]
        if missing:
            raise ValueError(f"Missing required vars: {missing}")
        
        # Build system prompt
        system = Template(self.base_system).substitute(merged)
        
        # Start with base prompt
        sections = [Template(self.base_prompt).substitute(merged)]
        
        # Add conditional sections
        claim_type = merged.get('claim_type', 'general')
        applied_sections = []
        
        for section in self.conditional_sections:
            if claim_type in section['applies_to'] or 'all' in section['applies_to']:
                # Only include section if required variables are present
                section_vars = section.get('requires_vars', [])
                if all(v in merged for v in section_vars):
                    sections.append(Template(section['content']).substitute(merged))
                    applied_sections.append(section['name'])
        
        # Add output schema
        sections.append(Template(self.output_schema).substitute(merged))
        
        prompt = "\n\n".join(sections)
        return prompt, system, applied_sections
    
    def run(self, max_tokens=3000, temperature=0.0, **kwargs):
        """Build and execute the conditional template."""
        prompt, system, applied_sections = self.build(**kwargs)
        result = ask(prompt, system=system, max_tokens=max_tokens, 
                     temperature=temperature)
        result['template_name'] = self.name
        result['claim_type'] = kwargs.get('claim_type', 'general')
        result['sections_applied'] = applied_sections
        return result

# --- Build a conditional claims analysis template ---
claims_analysis = ConditionalTemplate(
    name="claims_analysis",
    
    base_system="""You are a senior claims adjuster at Evergreen Mutual Insurance 
specializing in $claim_type claims. Analyze claims thoroughly using step-by-step 
reasoning. Always flag missing information rather than assuming.

GUARDRAILS:
- Ground analysis in provided policy and claim details
- Include disclaimer: "Preliminary analysis — subject to formal review"
- Never guarantee outcomes""",
    
    base_prompt="""Analyze this $claim_type insurance claim.

CLAIM DETAILS:
$claim_text""",
    
    conditional_sections=[
        # Auto-specific sections
        {
            'name': 'vehicle_analysis',
            'applies_to': ['auto'],
            'content': """VEHICLE INFORMATION:
$vehicle_info

Analyze vehicle damage severity, repair vs. total loss threshold, and 
whether the damage is consistent with the reported accident."""
        },
        {
            'name': 'liability_analysis',
            'applies_to': ['auto'],
            'content': """LIABILITY ASSESSMENT:
Determine fault allocation based on the facts provided.
Consider: traffic laws, right-of-way, witness statements, police report.
If comparative negligence applies, estimate the split."""
        },
        
        # Homeowners-specific sections
        {
            'name': 'property_analysis',
            'applies_to': ['homeowners'],
            'content': """PROPERTY DETAILS:
$property_info

Analyze the damage in context of the property's age, construction, 
and maintenance history. Consider code upgrade requirements."""
        },
        {
            'name': 'coverage_form_analysis',
            'applies_to': ['homeowners'],
            'content': """COVERAGE FORM ANALYSIS:
Policy form: $policy_form
Evaluate which coverage sections apply (A: Dwelling, B: Other Structures, 
C: Personal Property, D: Loss of Use). Check for relevant endorsements 
and sublimits."""
        },
        
        # Workers comp-specific sections
        {
            'name': 'injury_analysis',
            'applies_to': ['workers_comp'],
            'content': """INJURY DETAILS:
$injury_info

Assess: injury causation (work-related?), treatment reasonableness, 
expected disability duration, and return-to-work potential."""
        },
        {
            'name': 'wage_calculation',
            'applies_to': ['workers_comp'],
            'requires_vars': ['wage_info'],
            'content': """WAGE & BENEFIT CALCULATION:
$wage_info

Calculate: average weekly wage, compensation rate (apply state max/min), 
TTD duration and cost, potential PPD exposure."""
        },
        
        # Sections that apply to ALL claim types
        {
            'name': 'fraud_indicators',
            'applies_to': ['all'],
            'content': """FRAUD SCREENING:
Review the claim for any red flags or inconsistencies.
Check timing, narrative consistency, and documentation completeness."""
        },
        {
            'name': 'prior_claims',
            'applies_to': ['all'],
            'requires_vars': ['claims_history'],
            'content': """PRIOR CLAIMS HISTORY:
$claims_history

Evaluate claim frequency patterns and any relationship to current claim."""
        }
    ],
    
    output_schema="""\nProvide your analysis in these sections:

<reasoning>
Step-by-step analysis following your methodology.
</reasoning>

<determination>
Your preliminary coverage/liability determination.
</determination>

Then return a JSON summary (no markdown fencing):
<claim_json>
{
  "claim_type": "$claim_type",
  "peril": "string",
  "coverage_applies": true/false,
  "key_finding": "string (one sentence)",
  "estimated_exposure": number,
  "deductible": number,
  "risk_flags": ["string"],
  "next_steps": ["string"],
  "confidence": "HIGH | MEDIUM | LOW"
}
</claim_json>""",
    
    required_vars=['claim_text', 'claim_type'],
    defaults={'claims_history': 'No prior claims on file.'}
)

# --- Demo 1: Auto Claim ---
print("=" * 70)
print("DEMO 1: Auto Claim (auto-specific sections activate)")
print("=" * 70)

result = claims_analysis.run(
    claim_type='auto',
    claim_text="""On March 1, 2026, insured was driving northbound on Highway 99 
in moderate rain. A vehicle in the oncoming lane crossed the center line and 
struck the insured's vehicle head-on. Insured's 2023 Honda Accord sustained 
major front-end damage. Airbags deployed. Insured transported to hospital with 
broken wrist and chest contusions. Other driver cited for crossing center line. 
Policy: Full coverage, $500 deductible, 100/300/100 liability limits.""",
    vehicle_info="""2023 Honda Accord EX-L, 18,000 miles
Pre-accident value: ~$28,000
Damage: Major front-end — hood, bumper, radiator, both headlights, frame check needed
Airbags deployed (adds ~$3,000 to repair)
Preliminary estimate: $14,500 (pending frame inspection)
Total loss threshold: typically 70-75% of value""",
    claims_history="One minor claim in 2024 — parking lot fender bender, $1,800 paid."
)

print(f"Sections applied: {result['sections_applied']}")
print(f"\n{result['text'][:500]}...")

# Parse JSON
json_match = re.search(r'<claim_json>(.*?)</claim_json>', result['text'], re.DOTALL)
if json_match:
    try:
        parsed = parse_llm_json(json_match.group(1))
        print(f"\n\nJSON SUMMARY:")
        print(f"  Peril: {parsed['peril']}")
        print(f"  Coverage: {parsed['coverage_applies']}")
        print(f"  Exposure: ${parsed['estimated_exposure']:,.2f}")
        print(f"  Confidence: {parsed['confidence']}")
    except:
        pass
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Demo 2: Homeowners Claim ---
print("\n" + "=" * 70)
print("DEMO 2: Homeowners Claim (homeowners-specific sections activate)")
print("=" * 70)

result = claims_analysis.run(
    claim_type='homeowners',
    claim_text="""Insured reports that during a severe thunderstorm on February 28, 
2026, a large oak tree in the neighbor's yard was struck by lightning and fell 
onto the insured's detached garage, collapsing the roof and destroying a riding 
lawn mower and workshop tools inside. The tree also damaged 30 feet of fence. 
No injuries. Insured has photos and a neighbor's statement confirming the 
lightning strike.""",
    property_info="""Single family home, built 2008, well-maintained
Detached garage: 24x24, standard construction
Fence: Cedar privacy fence, installed 2019
No prior structural issues""",
    policy_form='HO-3 Special Form, $450K dwelling, $45K other structures (10%), $225K personal property, $2,000 deductible'
)

print(f"Sections applied: {result['sections_applied']}")
print(f"\n{result['text'][:500]}...")

json_match = re.search(r'<claim_json>(.*?)</claim_json>', result['text'], re.DOTALL)
if json_match:
    try:
        parsed = parse_llm_json(json_match.group(1))
        print(f"\n\nJSON SUMMARY:")
        print(f"  Peril: {parsed['peril']}")
        print(f"  Coverage: {parsed['coverage_applies']}")
        print(f"  Exposure: ${parsed['estimated_exposure']:,.2f}")
        print(f"  Confidence: {parsed['confidence']}")
    except:
        pass
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Demo 3: Workers Comp Claim ---
print("\n" + "=" * 70)
print("DEMO 3: Workers Comp Claim (workers comp-specific sections activate)")
print("=" * 70)

result = claims_analysis.run(
    claim_type='workers_comp',
    claim_text="""Employee reports shoulder injury sustained on February 27, 2026 
while lifting heavy boxes in the warehouse. Employee felt a pop in right shoulder 
and experienced immediate pain. Reported to supervisor within 30 minutes. Went to 
urgent care same day — diagnosed with rotator cuff tear. MRI confirmed full 
thickness tear requiring surgical repair. Employee is right-hand dominant.""",
    injury_info="""Diagnosis: Full thickness rotator cuff tear (right shoulder)
ICD-10: M75.111 (Complete rotator cuff tear, right shoulder)
Treatment plan: Arthroscopic repair surgery recommended
Surgeon: Dr. Williams, Orthopedic Associates
Expected surgery date: March 15, 2026
Pre-surgery PT: 2 weeks (3x/week)
Post-surgery recovery: 4-6 months
Return to full duty: 6-9 months estimated""",
    wage_info="""Average weekly wage: $1,150
State compensation rate: 66.67% of AWW
State max weekly benefit: $1,050
State min weekly benefit: $250
Job classification: Warehouse worker — heavy lifting required"""
)

print(f"Sections applied: {result['sections_applied']}")
print(f"\n{result['text'][:500]}...")

json_match = re.search(r'<claim_json>(.*?)</claim_json>', result['text'], re.DOTALL)
if json_match:
    try:
        parsed = parse_llm_json(json_match.group(1))
        print(f"\n\nJSON SUMMARY:")
        print(f"  Peril: {parsed['peril']}")
        print(f"  Coverage: {parsed['coverage_applies']}")
        print(f"  Exposure: ${parsed['estimated_exposure']:,.2f}")
        print(f"  Confidence: {parsed['confidence']}")
    except:
        pass
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Show which sections activated ---
print("\n" + "=" * 70)
print("CONDITIONAL SECTIONS — WHAT ACTIVATED WHERE")
print("=" * 70)
print("""
  AUTO CLAIM activated:
    ✓ vehicle_analysis (auto-specific)
    ✓ liability_analysis (auto-specific)
    ✓ fraud_indicators (all claims)
    ✓ prior_claims (all claims — history was provided)

  HOMEOWNERS CLAIM activated:
    ✓ property_analysis (homeowners-specific)
    ✓ coverage_form_analysis (homeowners-specific)
    ✓ fraud_indicators (all claims)
    ✗ prior_claims (all claims — but no history provided, used default)

  WORKERS COMP activated:
    ✓ injury_analysis (workers comp-specific)
    ✓ wage_calculation (workers comp-specific)
    ✓ fraud_indicators (all claims)
    ✗ prior_claims (all claims — but no history provided, used default)

  Same template, different claim types → different analysis sections.
  The template adapts to what it's processing.
""")

In [ ]:
# =============================================================================
# Template Composition — Full Runtime Assembly Pipeline
# =============================================================================
# Bringing it all together: composable system prompts from Day 24 +
# template library from Cell 2 + conditional sections from Cell 3 +
# a pipeline that processes a claim end-to-end using multiple templates.
# =============================================================================

# --- A claim processing pipeline using the template library ---

class ClaimsPipeline:
    """
    Orchestrates multiple templates in sequence to process a claim.
    Each step's output feeds the next step's input — like an assembly
    line where each station adds something to the product.
    """
    
    def __init__(self, templates, system_prompts):
        self.templates = templates
        self.system_prompts = system_prompts
        self.results = {}
    
    def run_step(self, step_name, template_name, display=True, **kwargs):
        """Run a single pipeline step using a named template."""
        template = self.templates[template_name]
        result = template.run(**kwargs)
        self.results[step_name] = result
        
        if display:
            print(f"\n  Step: {step_name}")
            print(f"  Template: {template_name}")
            print(f"  Tokens: {result['input_tokens']} in / {result['output_tokens']} out")
        
        return result
    
    def get_total_tokens(self):
        """Sum tokens across all pipeline steps."""
        total_in = sum(r['input_tokens'] for r in self.results.values())
        total_out = sum(r['output_tokens'] for r in self.results.values())
        return total_in, total_out
    
    def get_total_cost(self, input_rate=3.0, output_rate=15.0):
        """Calculate total cost across all steps (per million tokens)."""
        total_in, total_out = self.get_total_tokens()
        return (total_in * input_rate + total_out * output_rate) / 1_000_000


# --- Register a new template for the pipeline: Claim Summary Report ---
register_template(PromptTemplate(
    name="claim_report",
    system_template="""You are a senior claims adjuster at $company preparing 
a formal claim summary report for management review.""",
    template_str="""Generate a formal claim summary report using all available 
information from the processing pipeline.

CLAIM DATA:
$claim_text

CLASSIFICATION RESULT:
$classification

FRAUD SCREENING RESULT:
$fraud_result

Produce a concise management-ready report covering:
1. Claim overview (2-3 sentences)
2. Classification and priority
3. Risk assessment findings
4. Recommended next steps
5. Estimated exposure

Return ONLY valid JSON, no markdown fencing:
{
  "report": {
    "title": "string",
    "date": "string",
    "claim_overview": "string",
    "classification_summary": "string",
    "risk_assessment": {
      "score": number,
      "level": "string",
      "key_findings": ["string"]
    },
    "recommended_actions": ["string"],
    "estimated_exposure": {
      "low": number,
      "high": number
    },
    "assigned_priority": "string",
    "requires_management_approval": true/false
  }
}""",
    required_vars=['claim_text', 'classification', 'fraud_result'],
    defaults={'company': 'Evergreen Mutual Insurance'}
))

# --- Run the full pipeline ---
print("=" * 70)
print("CLAIMS PROCESSING PIPELINE")
print("=" * 70)

incoming_claim = """
Policyholder: Sandra Nguyen, Policy HO-3, $550K dwelling, $2,000 deductible
Date of Loss: February 26, 2026
Reported: March 2, 2026 (4-day reporting delay)

Description: The insured reports returning home from a weekend trip to find 
a large section of her living room ceiling had collapsed. She states the 
cause was a water leak from the upstairs bathroom. Upon investigation, a 
cracked toilet supply line was found. Water had been running for an estimated 
2-3 days.

Damages:
- Living room ceiling and drywall: $12,000
- Hardwood floor replacement (living room): $9,500
- Upstairs bathroom floor repair: $4,200
- Damaged furniture (sofa, entertainment center): $6,800
- Electronics (TV, sound system): $3,500
- Water remediation and drying: $5,500
- Total estimate: $41,500

Additional: Policy was renewed 6 weeks ago with a coverage increase from 
$450K to $550K. Insured has one prior water damage claim from 2023 ($7,200 paid).
The cracked supply line appears to be original to the 2009 construction.
"""

pipeline = ClaimsPipeline(TEMPLATE_LIBRARY, PERSONAS)

# Step 1: Classification
print("\n" + "-" * 70)
print("STEP 1: CLASSIFICATION")
print("-" * 70)

step1 = pipeline.run_step(
    'classification', 'classify_claim',
    claim_text=incoming_claim,
    line_of_business='homeowners'
)
try:
    classification = parse_llm_json(step1['text'])
    print(f"  Result: {classification['classification']} | "
          f"Priority: {classification['priority']} | "
          f"Confidence: {classification['confidence']}")
except:
    classification = step1['text']
    print(f"  Result: {step1['text'][:200]}")

# Step 2: Fraud Screening
print("\n" + "-" * 70)
print("STEP 2: FRAUD SCREENING")
print("-" * 70)

step2 = pipeline.run_step(
    'fraud_screening', 'fraud_screening',
    claim_text=incoming_claim,
    line_of_business='homeowners',
    claimant_history="""Prior claims:
- 2023: Water damage claim, $7,200 paid (same property)
Policy renewed 6 weeks ago with $100K coverage increase ($450K → $550K)."""
)
try:
    fraud = parse_llm_json(step2['text'])
    print(f"  Risk Score: {fraud['risk_score']}/10 | "
          f"Level: {fraud['risk_level']} | "
          f"Action: {fraud['recommended_action']}")
    for ind in fraud.get('indicators_found', []):
        print(f"    • [{ind['severity']}] {ind['indicator']}")
except:
    fraud = step2['text']
    print(f"  Result: {step2['text'][:200]}")

# Step 3: Customer Communication
print("\n" + "-" * 70)
print("STEP 3: CUSTOMER COMMUNICATION")
print("-" * 70)

step3 = pipeline.run_step(
    'customer_email', 'customer_email',
    policyholder_name='Sandra Nguyen',
    claim_number='EM-CLM-2026-03890',
    claim_status='Received — Under Initial Review',
    context='Water damage claim from toilet supply line failure. 4-day reporting delay due to weekend trip. Adjuster being assigned.',
    email_type='initial acknowledgment'
)
print(f"  Email generated ({step3['output_tokens']} tokens)")

# Step 4: Management Report
print("\n" + "-" * 70)
print("STEP 4: MANAGEMENT REPORT")
print("-" * 70)

step4 = pipeline.run_step(
    'report', 'claim_report',
    claim_text=incoming_claim,
    classification=json.dumps(classification) if isinstance(classification, dict) else str(classification),
    fraud_result=json.dumps(fraud) if isinstance(fraud, dict) else str(fraud)
)
try:
    report = parse_llm_json(step4['text'])
    print(f"  Title: {report['report']['title']}")
    print(f"  Priority: {report['report']['assigned_priority']}")
    print(f"  Exposure: ${report['report']['estimated_exposure']['low']:,.0f} — "
          f"${report['report']['estimated_exposure']['high']:,.0f}")
    print(f"  Mgmt Approval: {report['report']['requires_management_approval']}")
    print(f"  Actions:")
    for action in report['report']['recommended_actions']:
        print(f"    → {action}")
except:
    print(f"  Result: {step4['text'][:300]}")

# --- Pipeline Summary ---
print("\n" + "=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)

total_in, total_out = pipeline.get_total_tokens()
total_cost = pipeline.get_total_cost()

print(f"""
  Steps completed: {len(pipeline.results)}
  
  Token Usage:
    {'Step':<25} {'Input':>8} {'Output':>8} {'Total':>8}
    {'-' * 52}""")

for step_name, result in pipeline.results.items():
    total = result['input_tokens'] + result['output_tokens']
    print(f"    {step_name:<25} {result['input_tokens']:>8} {result['output_tokens']:>8} {total:>8}")

print(f"""    {'-' * 52}
    {'TOTAL':<25} {total_in:>8} {total_out:>8} {total_in + total_out:>8}
  
  Estimated cost: ${total_cost:.4f} per claim
  At 500 claims/day: ${total_cost * 500:.2f}/day | ${total_cost * 500 * 30:.2f}/month
  
  Pipeline output destinations:
    Step 1 (Classification) → DynamoDB (claim record)
    Step 2 (Fraud Screening) → SIU queue if flagged
    Step 3 (Customer Email)  → Email system (pending approval)
    Step 4 (Mgmt Report)     → Management dashboard
""")

In [ ]:
# =============================================================================
# CELL 5: Prompt Templates & Variables — Summary
# =============================================================================
# Tying together everything from Day 25
# =============================================================================

print("=" * 70)
print("DAY 25: PROMPT TEMPLATES & VARIABLES — SUMMARY")
print("=" * 70)

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

WHAT WE BUILT TODAY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Cell 1: Basic Prompt Templates
  • F-strings → string.Template → PromptTemplate class
  • Progression from quick-and-dirty to production-ready
  • PromptTemplate adds: validation, defaults, metadata tracking
  • Same template, different variables → different outputs

Cell 2: Template Library
  • Four registered templates: classify, coverage, email, fraud
  • register_template() pattern — like an MCP registry
  • Any developer calls by name, no prompt knowledge needed
  • Each template self-contained: system prompt + user prompt + schema

Cell 3: Conditional Templates
  • Dynamic sections activate based on claim type
  • Auto → vehicle + liability sections
  • Homeowners → property + coverage form sections
  • Workers comp → injury + wage calculation sections
  • Fraud screening activates for ALL claim types
  • Discussion: subclasses vs conditional for production safety

Cell 4: Pipeline Composition
  • Four templates chained: classify → fraud → email → report
  • Each step's output feeds the next step's input
  • Token and cost tracking across the full pipeline
  • One claim → four destinations (DB, SIU, email, dashboard)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

TEMPLATE ARCHITECTURE (PRODUCTION)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ┌─────────────────────────────────────────────────────┐
  │  DYNAMODB: Template Store                           │
  │  ├── System prompt components (Day 24)              │
  │  │   ├── Personas (versioned)                       │
  │  │   ├── Scopes (versioned)                         │
  │  │   └── Guardrails (versioned)                     │
  │  ├── Prompt templates (Day 25)                      │
  │  │   ├── classify_claim v2.1                        │
  │  │   ├── coverage_determination v1.4                │
  │  │   ├── customer_email v3.0                        │
  │  │   └── fraud_screening v1.2                       │
  │  └── Pipeline definitions                           │
  │      ├── standard_claim_pipeline                    │
  │      └── priority_claim_pipeline                    │
  └─────────────────────────────────────────────────────┘
           │
           ▼
  ┌─────────────────────────────────────────────────────┐
  │  LAMBDA: Template Engine                            │
  │  1. Receive claim event                             │
  │  2. Look up pipeline definition                     │
  │  3. Assemble system prompt from components          │
  │  4. Fill templates with claim data                  │
  │  5. Call Bedrock for each step                      │
  │  6. Validate output (Pydantic)                      │
  │  7. Route results to destinations                   │
  └─────────────────────────────────────────────────────┘
           │
           ▼
  ┌─────────────────────────────────────────────────────┐
  │  OUTPUT ROUTING                                     │
  │  ├── DynamoDB: claim record + classification        │
  │  ├── SQS: SIU queue (if fraud flagged)              │
  │  ├── SES: customer email (pending approval)         │
  │  ├── S3: audit trail (reasoning logs)               │
  │  └── Dashboard: management report                   │
  └─────────────────────────────────────────────────────┘

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

KEY CLASSES (carry these forward)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  PromptTemplate        — Reusable template with validation and defaults
  ConditionalTemplate   — Dynamic sections based on claim type
  ClaimsPipeline        — Multi-step orchestration with cost tracking
  TEMPLATE_LIBRARY      — Registry of named templates (dict)

  Utility functions:
  ask()                 — Bedrock Converse API wrapper
  parse_llm_json()      — Strip markdown fencing, return dict
  register_template()   — Add template to library

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DAYS 21-25: THE COMPLETE PROMPT ENGINEERING TOOLKIT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Day 21 — Prompt Fundamentals
    Zero-shot, few-shot, role-based, structured output
    → WHAT techniques exist

  Day 22 — Chain-of-Thought
    Step-by-step reasoning, structured CoT, auditable calculations
    → HOW the model thinks

  Day 23 — Output Formatting & Extraction
    Multi-doc extraction, templates, messy input, validation
    → WHAT the model produces and HOW we verify it

  Day 24 — System Prompt Engineering
    Persona, scope, guardrails, composability, boundary testing
    → HOW we configure the model's behavior

  Day 25 — Prompt Templates & Variables
    Template engine, library, conditional sections, pipelines
    → HOW we make everything reusable and production-ready

  Coming up:
  Day 26 — Evaluation Framework
    → HOW we measure whether all of this actually works

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")
```

Good commit message for Day 25:
```
Add Day 25 prompt templates notebook — template engine, library registry, conditional sections, and claims pipeline